# Manual annotation of `analyze_audio.py` pipeline outputs

Per-clip review tool: listen to the source audio, see every backend's output side-by-side (raw_16k + enhanced_16k together), inspect the per-clip aggregate report (`summary.json`, `disagreements.json`, `timeline.png`), and record one judgment per clip into a persistent JSON file in your home directory.

## Setup

1. Run on an HPC node so the audio files are reachable without download (the kernel reads the WAV; the browser receives an HTML5 audio widget).
2. Required packages: `ipywidgets`, `pillow` (for the timeline PNG). If missing: `pip install ipywidgets pillow` inside the project venv.
3. Edit `SUBJECT` + `DATASET_ROOT` in the config cell below if you want a different subject.
4. Annotations land in `~/senselab-annotations/<subject>.json` and persist across notebook sessions. The widget auto-jumps to the next unannotated clip on each load.
5. Re-run `scripts/annotation_summary.py` from a shell to aggregate annotations into stats / CSV when you're done.

In [7]:
# ─── Config ─────────────────────────────────────────────────────
from pathlib import Path

SUBJECT = "sub-ff75b163-af6b-4ff5-994c-b2d07a61c36a"
DATASET_ROOT = Path("/orcd/data/satra/002/datasets/b2aivoice/post_3.0/v3.1/adult/bids_registered_04_14_26/")
ANNOTATIONS_DIR = Path.home() / "senselab-annotations"
ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATIONS_FILE = ANNOTATIONS_DIR / f"{SUBJECT}.json"

print(f"Subject:           {SUBJECT}")
print(f"Dataset root:      {DATASET_ROOT}")
print(f"Annotations file:  {ANNOTATIONS_FILE}")

Subject:           sub-ff75b163-af6b-4ff5-994c-b2d07a61c36a
Dataset root:      /orcd/data/satra/002/datasets/b2aivoice/post_3.0/v3.1/adult/bids_registered_04_14_26
Annotations file:  /home/wilke18/senselab-annotations/sub-ff75b163-af6b-4ff5-994c-b2d07a61c36a.json


In [8]:
# ─── Discovery ──────────────────────────────────────────────────
# Each audio file may have multiple timestamped run_dirs (re-runs of
# analyze_audio.py); we keep only the LATEST per audio stem so the user
# always reviews the most recent pipeline output.

import re
from collections import Counter
from dataclasses import dataclass


@dataclass
class Clip:
    clip_id: str       # stable id == audio stem (no timestamp suffix)
    audio_path: Path
    run_dir: Path
    timestamp: str
    task_type: str
    session: str


_RUN_RE = re.compile(r"^(?P<stem>.+)_(?P<ts>\d{8}-\d{6})$")
_TASK_RE = re.compile(r"_task-(.+)$")


def discover_clips(subject: str) -> list[Clip]:
    """Return the latest run_dir per audio file under the subject's derivatives tree."""
    deriv_root = DATASET_ROOT / "derivatives" / "analyze_audio" / subject
    if not deriv_root.is_dir():
        return []
    latest: dict[str, Clip] = {}
    for run_dir in deriv_root.glob("ses-*/*"):
        if not run_dir.is_dir():
            continue
        m = _RUN_RE.match(run_dir.name)
        if not m:
            continue
        stem, ts = m.group("stem"), m.group("ts")
        if stem in latest and ts <= latest[stem].timestamp:
            continue
        session = run_dir.parent.name
        audio_path = DATASET_ROOT / subject / session / "audio" / f"{stem}.wav"
        tm = _TASK_RE.search(stem)
        task_type = re.sub(r"-\d+$", "", tm.group(1)) if tm else "?"
        latest[stem] = Clip(stem, audio_path, run_dir, ts, task_type, session)
    return sorted(latest.values(), key=lambda c: c.clip_id)


clips = discover_clips(SUBJECT)
print(f"Discovered {len(clips)} clips (latest run per audio file)\n")
print("Task type distribution:")
for t, n in Counter(c.task_type for c in clips).most_common():
    print(f"  {n:3d}  {t}")

Discovered 36 clips (latest run per audio file)

Task type distribution:
    6  Cape-V-sentences
    4  Respiration-and-cough-FiveBreaths
    3  Free-speech
    3  Maximum-phonation-time
    2  Respiration-and-cough-Breath
    2  Respiration-and-cough-Cough
    2  Respiration-and-cough-ThreeQuickBreaths
    1  Caterpillar-Passage
    1  Diadochokinesis-KA
    1  Diadochokinesis-PA
    1  Diadochokinesis-Pataka
    1  Diadochokinesis-TA
    1  Diadochokinesis-buttercup
    1  Free-Speech
    1  Glides-High-to-Low
    1  Glides-Low-to-High
    1  Loudness
    1  Picture-description
    1  Prolonged-vowel
    1  Rainbow-Passage
    1  Story-recall


In [9]:
# ─── Annotation storage ─────────────────────────────────────────
# One JSON file per subject in ~/senselab-annotations/. Atomic writes
# (write to .tmp, rename) so the file is never partially corrupted if
# the notebook kernel dies mid-save.

import json
from datetime import datetime


def load_annotations() -> dict:
    if not ANNOTATIONS_FILE.is_file():
        return {}
    return json.loads(ANNOTATIONS_FILE.read_text())


def save_annotation(clip_id: str, **fields) -> None:
    data = load_annotations()
    existing = data.get(clip_id, {})
    existing.update({k: v for k, v in fields.items() if v is not None or k == "pii_real"})
    existing["last_updated"] = datetime.utcnow().isoformat(timespec="seconds") + "Z"
    data[clip_id] = existing
    tmp = ANNOTATIONS_FILE.with_suffix(".tmp")
    tmp.write_text(json.dumps(data, indent=2, sort_keys=True))
    tmp.replace(ANNOTATIONS_FILE)


_annotations = load_annotations()
print(f"{len(_annotations)} existing annotations in {ANNOTATIONS_FILE.name} "
      f"({len(clips) - len(_annotations)} clips left to annotate)")

0 existing annotations in sub-ff75b163-af6b-4ff5-994c-b2d07a61c36a.json (36 clips left to annotate)


In [10]:
# ─── Per-clip JSON readers ──────────────────────────────────────
# Every stage JSON is wrapped by analyze_audio.py's `run_task_cached`
# in an outcome envelope: `{status, result, provenance, cache, ...}`.
# The actual payload lives under `result`. AST and YAMNet are
# windowed by analyze_audio.py (--ast-win-length / --yamnet-win-length)
# so their `result` is List[List[Dict]] — one window-list per audio,
# each window a {start, end, labels[], scores[]} dict.

from collections import defaultdict
from typing import Any

ASR_BACKEND_LABELS = {
    "openai_whisper_large_v3_turbo": "whisper",
    "ibm_granite_granite_speech_3_3_8b": "granite",
    "nvidia_canary_qwen_2_5b": "canary",
    "Qwen_Qwen3_ASR_1_7B": "qwen3",
}
DIAR_BACKEND_LABELS = {
    "pyannote_speaker_diarization_community_1": "pyannote",
    "nvidia_diar_sortformer_4spk_v1": "sortformer",
}
PASSES = ("raw_16k", "enhanced_16k")


def _load_json(p: Path) -> Any:
    try:
        return json.loads(p.read_text())
    except (FileNotFoundError, json.JSONDecodeError):
        return None


def _unwrap_outcome(d: Any) -> Any:
    """Peel off run_task_cached's outcome envelope."""
    if not isinstance(d, dict) or "status" not in d or "result" not in d:
        return d
    if d.get("status") != "ok":
        return f"⚠️ status={d.get('status')!r}"
    return d["result"]


def _walk_texts(x: Any):
    """Yield every `text` string nested anywhere inside a ScriptLine-ish tree."""
    if isinstance(x, dict):
        t = x.get("text")
        if isinstance(t, str):
            yield t
        for v in x.values():
            if isinstance(v, (list, dict)) and v is not t:
                yield from _walk_texts(v)
    elif isinstance(x, list):
        for item in x:
            yield from _walk_texts(item)


def _extract_text(d: Any) -> str:
    if d is None:
        return "⚠️ missing"
    inner = _unwrap_outcome(d)
    if isinstance(inner, str):
        return inner
    texts = [t for t in _walk_texts(inner) if t]
    if not texts:
        return ""
    deduped = [texts[0]]
    for t in texts[1:]:
        if t != deduped[-1]:
            deduped.append(t)
    return " ".join(deduped).strip()


def asr_texts(run_dir: Path, pass_label: str) -> dict[str, str]:
    pass_dir = run_dir / pass_label / "asr"
    if not pass_dir.is_dir():
        return {}
    return {
        ASR_BACKEND_LABELS.get(f.stem, f.stem): _extract_text(_load_json(f))
        for f in sorted(pass_dir.glob("*.json"))
    }


def diar_speaker_counts(run_dir: Path, pass_label: str) -> dict[str, int | None]:
    pass_dir = run_dir / pass_label / "diarization"
    if not pass_dir.is_dir():
        return {}
    out: dict[str, int | None] = {}
    for f in sorted(pass_dir.glob("*.json")):
        inner = _unwrap_outcome(_load_json(f))
        speakers: set[str] = set()
        def _collect(x):
            if isinstance(x, dict):
                spk = x.get("speaker")
                if spk is not None:
                    speakers.add(str(spk))
                for v in x.values():
                    if isinstance(v, (list, dict)):
                        _collect(v)
            elif isinstance(x, list):
                for item in x:
                    _collect(item)
        _collect(inner)
        out[DIAR_BACKEND_LABELS.get(f.stem, f.stem)] = len(speakers) if inner is not None else None
    return out


def pii_view(run_dir: Path, pass_label: str) -> dict[str, Any]:
    d = _load_json(run_dir / pass_label / "pii.json") or {}
    return {
        "detector_used": d.get("detector_used"),
        "detection_confidence": d.get("detection_confidence"),
        "contains_pii": d.get("contains_pii"),
        "n_spans": d.get("n_spans"),
        "categories": d.get("categories"),
        "spans": [
            {k: s.get(k) for k in ("text", "category", "source", "score", "asr_model")}
            for s in (d.get("spans") or [])
        ],
        "failures": d.get("failures") or {},
    }


def top_audio_events(run_dir: Path, pass_label: str, key: str, n: int = 5) -> list[tuple[str, float]]:
    """Top labels across all windows of a windowed AST/YAMNet result.

    Aggregates by mean score across the windows where the label appeared.
    """
    inner = _unwrap_outcome(_load_json(run_dir / pass_label / f"{key}.json"))
    if inner is None or isinstance(inner, str):
        return []

    windows: list[dict] = []

    def _flatten(x):
        if isinstance(x, list):
            for item in x:
                _flatten(item)
        elif isinstance(x, dict):
            windows.append(x)

    _flatten(inner)

    score_sum: defaultdict[str, float] = defaultdict(float)
    count: defaultdict[str, int] = defaultdict(int)
    for w in windows:
        if "labels" in w and "scores" in w and isinstance(w["labels"], list) and isinstance(w["scores"], list):
            for lbl, sc in zip(w["labels"], w["scores"]):
                try:
                    score_sum[str(lbl)] += float(sc)
                    count[str(lbl)] += 1
                except (TypeError, ValueError):
                    pass
        elif "label" in w:
            try:
                score_sum[str(w["label"])] += float(w.get("score") or w.get("confidence") or 0.0)
                count[str(w["label"])] += 1
            except (TypeError, ValueError):
                pass

    return sorted(
        ((lbl, score_sum[lbl] / count[lbl]) for lbl in score_sum if count[lbl] > 0),
        key=lambda x: x[1],
        reverse=True,
    )[:n]


def summary_headline(run_dir: Path) -> dict[str, Any]:
    """Extract the per-clip aggregate report from summary.json.

    analyze_audio.py writes the four-claim uncertainty (presence /
    identity / utterance / pii) under ``global_uncertainty.by_pass.<pl>``,
    not at the top level. Top-level we surface combined_uncertainty,
    best_pass, and any incomparable_reasons. Per-pass we surface each
    axis's status + combined_uncertainty so the reviewer can see at a
    glance which axis is driving the per-clip risk score.
    """
    d = _load_json(run_dir / "summary.json")
    if d is None:
        return {"⚠️": "summary.json not found"}
    gu = d.get("global_uncertainty") or {}
    out: dict[str, Any] = {
        "combined_uncertainty": gu.get("combined_uncertainty"),
        "best_pass": gu.get("best_pass"),
    }
    incomparable = gu.get("incomparable_reasons")
    if incomparable:
        out["incomparable_reasons"] = incomparable
    by_pass = gu.get("by_pass") or {}
    for pl in PASSES:
        ps = by_pass.get(pl)
        if not isinstance(ps, dict):
            continue
        out[pl] = {
            "combined_uncertainty": ps.get("combined_uncertainty"),
        }
        for axis in ("presence", "identity", "utterance", "pii"):
            ax = ps.get(axis)
            if isinstance(ax, dict):
                # Keep only the small interpretable fields, drop nested
                # vote tables and span detail.
                out[pl][axis] = {
                    k: ax.get(k)
                    for k in ("uncertainty", "claim", "n_voters", "confidence", "agreement", "decision")
                    if k in ax
                }
    # If global_uncertainty was never written (older run before comparator stage),
    # at least show the per-pass pipeline status so the reviewer knows what ran.
    if not gu and "passes" in d:
        out["passes_status"] = {pl: (d["passes"].get(pl) or {}).get("status") for pl in PASSES}
    return out


def disagreements_summary(run_dir: Path) -> dict[str, Any]:
    d = _load_json(run_dir / "disagreements.json") or {}
    if isinstance(d, dict):
        return {k: d[k] for k in list(d)[:12]}
    return {}


def timeline_paths(run_dir: Path) -> list[Path]:
    """Every plot file at the run_dir top level. analyze_audio.py writes a
    single `timeline.png` on recent runs and `timeline_001.png` /
    `timeline_002.png` on older ones — surface both so the reviewer always
    sees whatever is there."""
    return sorted(
        list(run_dir.glob("timeline*.png"))
        + list(run_dir.glob("*.png"))
    )


# ─── ASR consensus diff (on-demand) ─────────────────────────────
# Token-level visualization across every transcript (4 backends ×
# 2 passes = 8). Each word coloured by how many of the eight
# transcripts contain it: green = all, yellow = at least half,
# red = minority/unique.

import html as _html
from collections import Counter as _Counter


def make_consensus_html(transcripts: list[tuple[str, str]]) -> str:
    token_lists: list[tuple[str, list[str]]] = []
    for label, text in transcripts:
        toks = re.findall(r"[\w']+", (text or "").lower())
        token_lists.append((label, toks))
    in_count: _Counter = _Counter()
    for _, toks in token_lists:
        for t in set(toks):
            in_count[t] += 1
    n = max(1, len(token_lists))
    css = {
        "all":   "background:#a3e4a1;",
        "major": "background:#fff2a8;",
        "minor": "background:#ffb3b3;",
    }
    legend = (
        '<div style="margin:6px 0; font-family:sans-serif; font-size:12px; color:#444;">'
        '<b>Token consensus:</b> '
        f'<span style="{css["all"]} padding:1px 6px; border-radius:3px;">in every transcript</span> '
        f'<span style="{css["major"]} padding:1px 6px; border-radius:3px;">≥ half</span> '
        f'<span style="{css["minor"]} padding:1px 6px; border-radius:3px;">&lt; half</span>'
        '</div>'
    )
    rows: list[str] = []
    for label, toks in token_lists:
        spans = []
        for t in toks:
            c = in_count[t]
            style = css["all"] if c == n else (css["major"] if c * 2 >= n else css["minor"])
            spans.append(
                f'<span style="{style} padding:1px 4px; margin:1px; border-radius:3px;">'
                f'{_html.escape(t)}</span>'
            )
        line = " ".join(spans) if spans else '<i style="color:#888;">(empty)</i>'
        rows.append(
            f'<div style="margin:4px 0;">'
            f'<span style="display:inline-block; min-width:290px; font-family:monospace; '
            f'font-size:13px; color:#444;">{_html.escape(label)}</span> '
            f'{line}</div>'
        )
    return legend + '<div style="font-family:sans-serif; line-height:1.9;">' + "\n".join(rows) + '</div>'


In [11]:
# ─── Helpers layered on top of the readers cell ────────────────
# Small additions/overrides so we don't have to re-supply the big
# readers cell each time.

# 1) dedup timeline_paths — the first definition combined
# `glob("timeline*.png")` and `glob("*.png")` and double-counted
# timeline-prefixed files. Set-based version returns each PNG once.
def timeline_paths(run_dir: Path) -> list[Path]:
    return sorted(set(run_dir.glob("*.png")))


# 2) _fmt_text disambiguation: replace the bare U+2026 "…" suffix
# with an explicit `[+N chars truncated by notebook]` marker so an
# ellipsis in the table is unambiguously from the ASR model itself
# (Granite hitting max_new_tokens, Canary-Qwen emitting "I lik…")
# rather than from notebook display truncation. Cap raised to 800
# so the override only fires on genuinely long transcripts.
def _fmt_text(t: str, n: int = 800) -> str:
    t = (t or "").replace("|", "│").replace("\n", " ")
    if not t:
        return "_(empty)_"
    if len(t) > n:
        return f"{t[:n]} `[+{len(t) - n} chars truncated by notebook]`"
    return t


# 3) Fix: stop _walk_texts from descending into a ScriptLine's `chunks`.
#
# Whisper and Qwen3-ASR emit ScriptLine objects whose `text` is the
# full transcript AND whose `chunks` is the same content broken into
# per-word or per-segment dicts (also with `text` fields). The old
# walker yielded every nested `text`, so the joined output was
# "full transcript word1 word2 ..." — the transcript repeated
# effectively twice. The old `deduped` step only collapsed CONSECUTIVE
# duplicates, so word-by-word fragments slipped through.
#
# Rule: a dict that already has its own string `text` field is treated
# as a self-contained transcript-bearing unit. We yield that text and
# do NOT recurse into its children. Sibling list / dict structures
# that don't themselves have `text` are still traversed (so envelope
# wrappers and List[List[ScriptLine]] still work).
def _walk_texts(x):
    if isinstance(x, dict):
        t = x.get("text")
        if isinstance(t, str):
            yield t
            return  # don't descend into chunks/segments — same content, just timed
        for v in x.values():
            if isinstance(v, (list, dict)):
                yield from _walk_texts(v)
    elif isinstance(x, list):
        for item in x:
            yield from _walk_texts(item)


def _extract_text(d):
    if d is None:
        return "⚠️ missing"
    inner = _unwrap_outcome(d)
    if isinstance(inner, str):
        return inner
    texts = [t for t in _walk_texts(inner) if t]
    if not texts:
        return ""
    deduped = [texts[0]]
    for t in texts[1:]:
        if t != deduped[-1]:
            deduped.append(t)
    return " ".join(deduped).strip()


# 4) audio_quality_metrics: pull per-window-mean values from the
# features parquets so the reviewer sees torchaudio_squim's STOI /
# PESQ / SI-SDR predictions and parselmouth's HNR / jitter / shimmer
# at a glance before rating audio quality. Pandas is a senselab core
# dep so the import is safe; the function is defensively shaped so a
# missing parquet, missing column, or missing pandas just yields {}.
def audio_quality_metrics(run_dir: Path, pass_label: str) -> dict[str, dict[str, float]]:
    """Return {backend: {metric: mean_value}} for any audio-quality columns found."""
    feat_dir = run_dir / pass_label / "features"
    out: dict[str, dict[str, float]] = {}
    if not feat_dir.is_dir():
        return out
    try:
        import pandas as pd
    except ImportError:
        return out

    def _mean_numeric(path: Path, name_filter=None) -> dict[str, float]:
        if not path.is_file():
            return {}
        try:
            df = pd.read_parquet(path)
        except Exception:  # noqa: BLE001
            return {}
        result: dict[str, float] = {}
        for col in df.select_dtypes(include="number").columns:
            if col in ("start", "end"):  # window boundaries, not quality
                continue
            if name_filter is not None and not name_filter(col):
                continue
            try:
                v = float(df[col].mean())
                if v == v:  # filter NaN
                    result[str(col)] = v
            except Exception:  # noqa: BLE001
                continue
        return result

    squim = _mean_numeric(feat_dir / "torchaudio_squim.parquet")
    if squim:
        out["torchaudio_squim"] = squim

    def _is_voice_quality(col: str) -> bool:
        c = col.lower()
        return any(s in c for s in ("hnr", "jitter", "shimmer"))

    praat = _mean_numeric(feat_dir / "parselmouth.parquet", _is_voice_quality)
    if praat:
        out["parselmouth"] = praat

    return out


In [12]:
# ─── Annotation widget — slim layout ────────────────────────────
# Audio + timeline drive every judgment so they sit at the top. Every
# other section is the minimum data the reviewer needs to fill in the
# adjacent form field, no more. Verbose JSON dumps and debug paths
# are intentionally absent — the CLI script aggregates the pipeline's
# own automated judgments separately.

import ipywidgets as widgets
from IPython.display import Audio, HTML, Image, Markdown, display

_FORM_STYLE = {"description_width": "180px"}


def _first_unannotated_idx() -> int:
    done = set(load_annotations())
    for i, c in enumerate(clips):
        if c.clip_id not in done:
            return i
    return 0


# ─── Form widgets ───────────────────────────────────────────────
audio_quality_rating = widgets.IntSlider(
    value=3, min=1, max=5, step=1,
    description="Audio quality (1-5):", style=_FORM_STYLE,
)
asr_rating = widgets.IntSlider(
    value=3, min=1, max=5, step=1,
    description="ASR (majority) (1-5):", style=_FORM_STYLE,
)
diar_voices = widgets.IntText(
    value=1, description="# distinct voices:", style=_FORM_STYLE,
)
pii_real = widgets.RadioButtons(
    options=[("True PII present", True), ("All false positives", False), ("No PII flagged / N/A", None)],
    description="PII assessment:", style=_FORM_STYLE,
)
ast_yamnet_unexpected = widgets.Checkbox(
    value=False, description="Something unexpected in AST/YAMNet output", indent=False,
)
notes = widgets.Textarea(
    description="Notes:", style=_FORM_STYLE,
    layout={"width": "700px", "height": "40px"},  # half the previous height
)

# ─── Navigation widgets ─────────────────────────────────────────
save_btn = widgets.Button(description="Save + next ▶", button_style="success")
save_only_btn = widgets.Button(description="Save (stay)", button_style="info")
prev_btn = widgets.Button(description="◀ Prev")
next_btn = widgets.Button(description="Next ▶")
consensus_btn = widgets.Button(description="Show ASR consensus", icon="search-plus")
consensus_clear_btn = widgets.Button(description="Hide consensus", icon="times")
goto = widgets.BoundedIntText(value=0, min=0, max=max(0, len(clips) - 1), description="Goto idx:", style=_FORM_STYLE)
filter_unannotated = widgets.Checkbox(value=False, description="Skip already-annotated on next/prev")
task_filter = widgets.Dropdown(
    options=["(all)"] + sorted({c.task_type for c in clips}),
    description="Task filter:", style=_FORM_STYLE,
)

# ─── Timeline navigation ────────────────────────────────────────
timeline_select = widgets.Dropdown(options=[], description="Timeline:", style=_FORM_STYLE)
timeline_prev_btn = widgets.Button(description="◀", layout={"width": "40px"})
timeline_next_btn = widgets.Button(description="▶", layout={"width": "40px"})
timeline_label = widgets.Label(value="")

# ─── Per-section data views ─────────────────────────────────────
_header_view = widgets.Output()
_timeline_view = widgets.Output()
_audio_quality_view = widgets.Output()
_asr_view = widgets.Output()
_consensus_view = widgets.Output()
_diar_view = widgets.Output()
_pii_data_view = widgets.Output()
_ast_view = widgets.Output()

_state = {"idx": _first_unannotated_idx()}


# ─── Per-section render functions ───────────────────────────────


def _render_header(clip: Clip, idx: int, pool_size: int, annotated_total: int) -> None:
    _header_view.clear_output()
    with _header_view:
        display(Markdown(
            f"`[{idx + 1} / {len(clips)}]` · annotated: **{annotated_total}** · pool: **{pool_size}**  \n"
            f"**`{clip.clip_id}`** · task=`{clip.task_type}`"
        ))
        if clip.audio_path.is_file():
            display(Audio(str(clip.audio_path), autoplay=False))
        else:
            display(Markdown(f"⚠️ Audio file not found at `{clip.audio_path}`"))


def _render_timeline(_change=None) -> None:
    _timeline_view.clear_output()
    opts = [v for _, v in (timeline_select.options or [])]
    if not opts:
        timeline_label.value = ""
        with _timeline_view:
            display(Markdown("_(no timeline.png in this run_dir)_"))
        return
    cur = timeline_select.value
    if cur not in opts:
        cur = opts[0]
        timeline_select.value = cur
    timeline_label.value = f"{opts.index(cur) + 1} / {len(opts)}"
    with _timeline_view:
        if Path(cur).is_file():
            display(Image(str(cur)))
        else:
            display(Markdown(f"⚠️ Timeline file vanished: `{cur}`"))


def _timeline_step(delta: int) -> None:
    opts = [v for _, v in (timeline_select.options or [])]
    if not opts:
        return
    cur = timeline_select.value
    if cur not in opts:
        timeline_select.value = opts[0]
        return
    timeline_select.value = opts[(opts.index(cur) + delta) % len(opts)]


# Audio-quality metric whitelist — only the interpretable ones, in
# the order a clinician would read them. Substrings, not exact matches,
# so column-name drift across pandas / parselmouth / torchaudio_squim
# versions doesn't silently drop a metric.
_AQ_KEY_METRICS = ("stoi", "pesq", "sisdr", "si_sdr", "hnr", "jitter", "shimmer")


def _render_audio_quality(clip: Clip) -> None:
    _audio_quality_view.clear_output()
    per_pass = {p: audio_quality_metrics(clip.run_dir, p) for p in PASSES}
    # Flatten every (backend, metric) to a single key; preserve the
    # _AQ_KEY_METRICS ordering and skip the rest.
    pairs: list[tuple[str, str]] = []  # (display_name, full_key)
    for backend, sub in (per_pass.get("raw_16k") or {}).items():
        for metric in sub:
            ml = metric.lower()
            if any(k in ml for k in _AQ_KEY_METRICS):
                pairs.append((metric, f"{backend}|{metric}"))
    # Stable order: order of _AQ_KEY_METRICS, then alphabetical within
    pairs.sort(key=lambda p: (next((i for i, k in enumerate(_AQ_KEY_METRICS) if k in p[0].lower()), 999), p[0]))
    with _audio_quality_view:
        if not pairs:
            display(Markdown("_(no audio-quality metrics found)_"))
            return
        rows = ["| metric | raw → enhanced |", "|---|---|"]
        for name, full in pairs:
            backend, metric = full.split("|", 1)
            raw_v = (per_pass["raw_16k"].get(backend) or {}).get(metric)
            enh_v = (per_pass["enhanced_16k"].get(backend) or {}).get(metric)
            def _f(v): return f"{v:.3f}" if isinstance(v, float) else "?"
            arrow = "→"
            if isinstance(raw_v, float) and isinstance(enh_v, float):
                if abs(enh_v - raw_v) < 1e-9:
                    arrow = "="
                elif enh_v > raw_v:
                    arrow = "↑"
                else:
                    arrow = "↓"
            rows.append(f"| `{name}` | {_f(raw_v)} {arrow} {_f(enh_v)} |")
        display(Markdown("\n".join(rows)))


def _render_asr(clip: Clip) -> None:
    _asr_view.clear_output()
    _consensus_view.clear_output()
    with _asr_view:
        asr = {p: asr_texts(clip.run_dir, p) for p in PASSES}
        backends = sorted({b for p in PASSES for b in asr[p]})
        if not backends:
            display(Markdown("_(no ASR transcripts found)_"))
            return
        rows = ["| backend | raw_16k | enhanced_16k |", "|---|---|---|"]
        for b in backends:
            rows.append(
                f"| `{b}` | "
                f"{_fmt_text(asr['raw_16k'].get(b, '⚠️ missing'))} | "
                f"{_fmt_text(asr['enhanced_16k'].get(b, '⚠️ missing'))} |"
            )
        display(Markdown("\n".join(rows)))


def _render_diarization(clip: Clip) -> None:
    _diar_view.clear_output()
    with _diar_view:
        diar = {p: diar_speaker_counts(clip.run_dir, p) for p in PASSES}
        backends = sorted({b for p in PASSES for b in diar[p]})
        if not backends:
            display(Markdown("_(no diarization data)_"))
            return
        # One-line summary: pyannote: raw|enh  ·  sortformer: raw|enh
        parts = [
            f"`{b}`: {diar['raw_16k'].get(b, '?')} | {diar['enhanced_16k'].get(b, '?')}"
            for b in backends
        ]
        display(Markdown("**raw | enhanced** ·  " + "  ·  ".join(parts)))


def _render_pii(clip: Clip) -> None:
    _pii_data_view.clear_output()
    with _pii_data_view:
        pii = {p: pii_view(clip.run_dir, p) for p in PASSES}
        raw_spans = pii["raw_16k"].get("spans") or []
        enh_spans = pii["enhanced_16k"].get("spans") or []
        if not raw_spans and not enh_spans:
            # Nothing to review — keep it to one line, surface failure if any
            failures: list[str] = []
            for p in PASSES:
                f = pii[p].get("failures") or {}
                if f:
                    failures.append(f"{p}: {list(f)[0]}")
            line = "_no PII flagged across both passes_"
            if failures:
                line += f"  ·  ⚠️ failures: {'; '.join(failures)}"
            display(Markdown(line))
            return
        lines: list[str] = []
        for p in PASSES:
            v = pii[p]
            n = v.get("n_spans") or 0
            conf = v.get("detection_confidence")
            if n:
                lines.append(f"**{p}** — {n} spans · confidence=`{conf}`")
                for s in v.get("spans") or []:
                    lines.append(
                        f"  - `{s['text']!r}` → {s['category']} "
                        f"(`{s['source']}` {s['score']}, from `{s['asr_model']}`)"
                    )
            else:
                lines.append(f"**{p}** — _(none)_")
        display(Markdown("\n".join(lines)))


def _render_ast_yamnet(clip: Clip) -> None:
    _ast_view.clear_output()
    with _ast_view:
        rows = ["| stage | raw_16k | enhanced_16k |", "|---|---|---|"]
        any_row = False
        for stage in ("ast", "yamnet"):
            raw = top_audio_events(clip.run_dir, "raw_16k", stage, n=5) or []
            enh = top_audio_events(clip.run_dir, "enhanced_16k", stage, n=5) or []
            def _fmt(lst): return "<br>".join(f"{lbl} ({sc:.2f})" for lbl, sc in lst) or "_(none)_"
            if raw or enh:
                any_row = True
            rows.append(f"| {stage} | {_fmt(raw)} | {_fmt(enh)} |")
        display(Markdown("\n".join(rows) if any_row else "_(no AST / YAMNet data)_"))


# ─── Navigation + persistence ───────────────────────────────────


def _filtered_indexes() -> list[int]:
    idxs = list(range(len(clips)))
    if task_filter.value != "(all)":
        idxs = [i for i in idxs if clips[i].task_type == task_filter.value]
    if filter_unannotated.value:
        done = set(load_annotations())
        idxs = [i for i in idxs if clips[i].clip_id not in done]
    return idxs


def _step(delta: int) -> None:
    if not clips:
        return
    pool = _filtered_indexes()
    if not pool:
        return
    cur = _state["idx"]
    if cur not in pool:
        cur = pool[0]
    pos = pool.index(cur)
    pos = (pos + delta) % len(pool)
    _state["idx"] = pool[pos]
    refresh()


def refresh() -> None:
    if not clips:
        _header_view.clear_output()
        with _header_view:
            print("(no clips discovered)")
        return
    idx = _state["idx"] % len(clips)
    clip = clips[idx]
    pool_size = len(_filtered_indexes())
    annotated_total = len(load_annotations())
    _render_header(clip, idx, pool_size, annotated_total)
    tls = timeline_paths(clip.run_dir)
    new_opts = [(p.name, str(p)) for p in tls]
    timeline_select.unobserve(_render_timeline, names="value")
    timeline_select.options = new_opts
    if new_opts:
        timeline_select.value = new_opts[0][1]
    timeline_select.observe(_render_timeline, names="value")
    _render_timeline()
    _render_audio_quality(clip)
    _render_asr(clip)
    _render_diarization(clip)
    _render_pii(clip)
    _render_ast_yamnet(clip)
    cur = load_annotations().get(clip.clip_id, {})
    audio_quality_rating.value = cur.get("audio_quality_rating", 3)
    asr_rating.value = cur.get("asr_rating", 3)
    diar_voices.value = cur.get("diar_voices", 1)
    pii_real.value = cur.get("pii_real")
    raw_unexpected = cur.get("ast_yamnet_unexpected")
    ast_yamnet_unexpected.value = bool(raw_unexpected) if raw_unexpected is not None else False
    notes.value = cur.get("notes") or ""
    goto.value = idx


def _save_current() -> None:
    idx = _state["idx"] % len(clips)
    clip = clips[idx]
    save_annotation(
        clip.clip_id,
        audio_quality_rating=int(audio_quality_rating.value),
        asr_rating=int(asr_rating.value),
        diar_voices=int(diar_voices.value),
        pii_real=pii_real.value,
        ast_yamnet_unexpected=bool(ast_yamnet_unexpected.value),
        notes=notes.value.strip() or None,
        task_type=clip.task_type,
    )


def _show_consensus(_):
    if not clips:
        return
    idx = _state["idx"] % len(clips)
    clip = clips[idx]
    pairs: list[tuple[str, str]] = []
    for p in PASSES:
        for b, t in asr_texts(clip.run_dir, p).items():
            pairs.append((f"{b} · {p}", t or ""))
    _consensus_view.clear_output()
    with _consensus_view:
        if not pairs:
            display(Markdown("_(no ASR transcripts found for this clip)_"))
        else:
            display(HTML(make_consensus_html(pairs)))


def _hide_consensus(_):
    _consensus_view.clear_output()


def _on_goto(change):
    if change.get("name") == "value":
        _state["idx"] = int(change["new"]) % max(1, len(clips))
        refresh()


save_btn.on_click(lambda _: (_save_current(), _step(+1)))
save_only_btn.on_click(lambda _: (_save_current(), refresh()))
prev_btn.on_click(lambda _: _step(-1))
next_btn.on_click(lambda _: _step(+1))
consensus_btn.on_click(_show_consensus)
consensus_clear_btn.on_click(_hide_consensus)
timeline_prev_btn.on_click(lambda _: _timeline_step(-1))
timeline_next_btn.on_click(lambda _: _timeline_step(+1))
timeline_select.observe(_render_timeline, names="value")
goto.observe(_on_goto, names="value")
filter_unannotated.observe(lambda _: refresh(), names="value")
task_filter.observe(lambda _: refresh(), names="value")


# ─── Layout ─────────────────────────────────────────────────────
# Slimmed: no <h3> separators, no aggregate report, no disagreements,
# no debug paths, no "saved annotation" echo. Section titles are
# inline bold markdown in each section's first line.
def _bold(label: str) -> widgets.HTML:
    return widgets.HTML(value=f'<div style="font-weight:bold; margin-top:10px;">{label}</div>')


display(widgets.VBox([
    widgets.HBox([prev_btn, next_btn, goto, task_filter, filter_unannotated]),
    _header_view,
    widgets.HBox(
        [widgets.Label("Timeline:"), timeline_prev_btn, timeline_select, timeline_next_btn, timeline_label]
    ),
    _timeline_view,
    _bold("Audio quality"),
    _audio_quality_view,
    audio_quality_rating,
    _bold("ASR"),
    _asr_view,
    widgets.HBox([consensus_btn, consensus_clear_btn]),
    _consensus_view,
    asr_rating,
    _bold("Diarization"),
    _diar_view,
    diar_voices,
    _bold("PII"),
    _pii_data_view,
    pii_real,
    _bold("AST / YAMNet"),
    _ast_view,
    ast_yamnet_unexpected,
    notes,
    widgets.HBox([save_btn, save_only_btn]),
]))
refresh()


## Progress + export

From a shell, aggregate all subjects' annotations into stats / CSV:

```bash
python scripts/annotation_summary.py                    # prints summary across every annotated subject
python scripts/annotation_summary.py --subject sub-018js
python scripts/annotation_summary.py --csv annotations.csv
```